# 02 · Factorial HMM：先验独立，后验耦合
两个 emitter，各有独立 Markov 链，观测是它们的光子总和。联合状态为 (0,0),(0,1),(1,0),(1,1)。这是“factorial HMM”，不意味着 posterior 也因子化。相同 brightness 且无空间区分时有标签对称性。


In [ ]:
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
sys.path.insert(0,str(root))
import numpy as np
import matplotlib.pyplot as plt
from src.hmm import forward_backward, brute_force, poisson_log_emissions
rng=np.random.default_rng(7)


In [ ]:
from itertools import product
states=np.array(list(product([0,1],repeat=2)))
A1=np.array([[.9,.1],[.2,.8]]); A2=np.array([[.85,.15],[.3,.7]])
A=np.kron(A1,A2); pi=np.kron([.8,.2],[.7,.3])
rates=1+states@np.array([4.,4.])
y=np.array([1,5,4,8,2,0])
ll,g,_,_=forward_backward(poisson_log_emissions(y,rates),pi,A)
p1=g@states[:,0]; p2=g@states[:,1]
cov=g[:,3]-p1*p2
print('posterior covariance between emitters:',cov)
plt.plot(p1,label='emitter 1 on'); plt.plot(p2,label='emitter 2 on');
plt.xlabel('frame'); plt.ylabel('posterior probability'); plt.legend();


## 空间如何进入模型？
将单个 count 替换为像素向量：$y_{tp}\sim\mathrm{Poisson}(b_p+\sum_m a_m z_{tm}h_p(r_m))$。每个联合状态的 log emission 是各像素 log likelihood 之和。像素在给定状态下独立，边缘化状态后不独立。

M 个二状态 emitter 有 $2^M$ 联合状态。朴素 dense forward 为 $O(T4^M)$；Kronecker 结构可改善转移运算，但不能消除后验状态空间本身的增长。这里只取 M=2。

练习：brightness 改为 [3,6]；解释标签识别如何变化。再给两个 emitter 不同像素模式，写出 log emission。
